# Phase 1 — Architecture Validation on Synthetic StreamSpot Data

**Thesis:** Applying Causality Tracking and Incremental Alignment for Graph-Based Threat Hunting  
**Author:** Tran Thai Huy  
**Purpose:** Confirm the modified encoder-decoder architecture trains end-to-end before touching DARPA TC data.  
**Success criterion:** BCE + CE training loss decreases over 10 epochs.

---

## What this notebook validates

1. `NodeTypeProjection` correctly maps heterogeneous node features (4-8 dims) to 64-dim common space  
2. 1-layer GAT (4 heads × 16-dim) produces 64-dim spatial embeddings  
3. GRU cell (64-dim) maintains per-node temporal state  
4. Dual-head `EdgeDecoder` predicts edge existence + edge type  
5. Self-supervised loss (BCE + CE) decreases — proving the architecture can learn  
6. Per-edge anomaly scores separate attack graphs from benign graphs better than random  

**Data:** Synthetic graphs mimicking StreamSpot's structure (600 graphs, 6 node types, 9 edge types).  
StreamSpot is used as a sanity check only, not as primary evaluation (Manzoor et al., KDD 2016).

---
## Section 0 — Environment & Imports

In [1]:
import sys
import os
import json
import random
import math
from collections import Counter
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── PyG import guard (torch-sparse fails on Windows) ─────────────────────────
try:
    from torch_geometric.nn import GATConv
    from torch_geometric.data import Data
    PYGEOMETRIC_AVAILABLE = True
except ImportError:
    PYGEOMETRIC_AVAILABLE = False
    print("PyG not available — using fallback implementations")

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python:  {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU mode'})")
print(f"PyG:     {PYGEOMETRIC_AVAILABLE}")
print(f"Device:  {DEVICE}")
print(f"Seed:    {SEED}")
print()
print("This notebook validates the Ophanim-EDR causality engine architecture")
print("on synthetic StreamSpot-like data before moving to DARPA TC evaluation.")

PyG not available — using fallback implementations
Python:  3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
PyTorch: 2.11.0+cpu
CUDA:    False (CPU mode)
PyG:     False
Device:  cpu
Seed:    42

This notebook validates the Ophanim-EDR causality engine architecture
on synthetic StreamSpot-like data before moving to DARPA TC evaluation.


---
## Section 1 — Synthetic StreamSpot Data Generator

### Why synthetic data?

StreamSpot (Manzoor et al., KDD 2016) provides 600 information-flow graphs with graph-level
labels (benign/attack). We generate *synthetic* graphs matching its structure for two reasons:

1. **Isolation:** We want to validate that the architecture can learn *at all* before debugging
   data loading, format conversion, and preprocessing. If training loss doesn't decrease on
   clean synthetic data, the problem is in the model, not the data pipeline.

2. **Reproducibility:** Synthetic data is fully deterministic and doesn't require downloading
   external datasets. Anyone can reproduce this notebook from a clean clone.

### What the synthetic data looks like

Each graph represents a simulated system execution trace:
- **Nodes** are OS entities (processes, files, sockets, etc.) with type-specific features
- **Edges** are causal events (read, write, execute, etc.) between entities
- **Labels** are graph-level: 0 = benign, 1 = attack

Attack graphs differ from benign graphs in their *structure*: they contain more unusual
edge-type combinations given their node types (e.g., a socket node receiving EXECUTE —
impossible in benign traces but common in code injection). This conditional structure is what
the anomaly detector learns and what attacks break.


In [2]:
# ── CDM Schema Constants ─────────────────────────────────────────────────────
# These match causality-engine/graph/schema.py exactly.

NUM_NODE_TYPES = 6  # process, file, socket, registry, memory, other
NUM_EDGE_TYPES = 9  # WRITE, READ, EXECUTE, FORK_CLONE, CONNECT, SEND, RECEIVE, MMAP, RENAME_LINK
EMBEDDING_DIM = 64

# Raw feature dimensions per node type (before projection to common space).
# These reflect the telemetry fields available from Sysmon/ETW/auditd.
NODE_FEATURE_DIMS = {
    0: 8,   # process: pid, ppid, uid, gid, start_time, cmd_hash, privilege_level, session_id
    1: 4,   # file:    path_hash, size, permissions, inode
    2: 6,   # socket:  src_ip, src_port, dst_ip, dst_port, protocol, state
    3: 5,   # registry: hive, key_hash, value_hash, access_mask, data_type
    4: 4,   # memory:  base_addr, size, protection, mapped_file_hash
    5: 3,   # other:   type_indicator, timestamp, generic_hash
}
MAX_FEAT_DIM = max(NODE_FEATURE_DIMS.values())  # 8 — used for zero-padding

# Node type distribution in typical benign provenance graphs.
# Processes and files dominate; sockets and registry are less common.
BENIGN_TYPE_WEIGHTS = [0.30, 0.35, 0.15, 0.10, 0.05, 0.05]

# ── Structure-aware edge generation ──────────────────────────────────────────
# In real provenance graphs, edge types are NOT independent of node types.
# A process->file edge is almost always READ or WRITE, never CONNECT.
# A process->socket edge is almost always CONNECT, SEND, or RECEIVE.
#
# This table encodes P(edge_type | src_type, dst_type) for benign behaviour.
# Only the dominant pairs are listed; others use a uniform fallback.
#
# Edge types: 0=WRITE 1=READ 2=EXECUTE 3=FORK_CLONE 4=CONNECT 5=SEND 6=RECEIVE 7=MMAP 8=RENAME_LINK

BENIGN_EDGE_TYPE_PROBS = {
    # process -> file: mostly READ/WRITE, some EXECUTE/MMAP/RENAME
    (0, 1): [0.30, 0.35, 0.10, 0.00, 0.00, 0.00, 0.00, 0.15, 0.10],
    # process -> process: FORK_CLONE dominates, some EXECUTE
    (0, 0): [0.00, 0.00, 0.30, 0.60, 0.00, 0.00, 0.00, 0.00, 0.10],
    # process -> socket: networking operations
    (0, 2): [0.00, 0.00, 0.00, 0.00, 0.40, 0.35, 0.25, 0.00, 0.00],
    # process -> registry: READ/WRITE registry
    (0, 3): [0.45, 0.45, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.10],
    # process -> memory: MMAP dominates
    (0, 4): [0.10, 0.10, 0.00, 0.00, 0.00, 0.00, 0.00, 0.70, 0.10],
    # file -> process: EXECUTE (loading), READ (config)
    (1, 0): [0.00, 0.30, 0.50, 0.00, 0.00, 0.00, 0.00, 0.10, 0.10],
    # socket -> process: RECEIVE dominates (incoming data)
    (2, 0): [0.00, 0.00, 0.00, 0.00, 0.10, 0.05, 0.75, 0.00, 0.10],
    # file -> file: RENAME_LINK dominates
    (1, 1): [0.15, 0.15, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.70],
}

# Pairs that connect src_type->dst_type in benign graphs, with relative frequency.
# Edges predominantly flow from processes to other entities.
BENIGN_PAIR_WEIGHTS = {
    (0, 1): 0.30,   # process -> file (dominant)
    (0, 0): 0.15,   # process -> process (fork/exec)
    (0, 2): 0.12,   # process -> socket
    (0, 3): 0.08,   # process -> registry
    (0, 4): 0.05,   # process -> memory
    (1, 0): 0.12,   # file -> process (execution)
    (2, 0): 0.08,   # socket -> process (incoming)
    (1, 1): 0.05,   # file -> file (rename)
    (0, 5): 0.03,   # process -> other
    (5, 0): 0.02,   # other -> process
}

# Uniform fallback for edge types on unusual (src_type, dst_type) pairs
UNIFORM_EDGE_PROBS = [1.0 / NUM_EDGE_TYPES] * NUM_EDGE_TYPES


@dataclass
class GraphData:
    """Minimal graph container -- PyG-compatible fields without PyG dependency."""
    x: torch.Tensor           # [N, max_feat_dim] zero-padded node features
    edge_index: torch.Tensor  # [2, E] COO edge indices
    edge_type: torch.Tensor   # [E] integer edge type labels
    node_types: torch.Tensor  # [N] integer node type labels
    y: int                    # graph-level label: 0=benign, 1=attack
    num_nodes: int
    num_edges: int


def _sample_edge_type(src_type: int, dst_type: int, is_attack: bool) -> int:
    """Sample an edge type conditioned on (src_type, dst_type) pair."""
    pair = (src_type, dst_type)
    probs = list(BENIGN_EDGE_TYPE_PROBS.get(pair, UNIFORM_EDGE_PROBS))

    if is_attack:
        # Attack: flatten the distribution -- attacker uses unusual operations
        # e.g., WRITE where only READ is expected, EXECUTE from socket, etc.
        uniform = [1.0 / NUM_EDGE_TYPES] * NUM_EDGE_TYPES
        # 60% uniform + 40% benign -> breaks the learned pattern significantly
        probs = [0.6 * u + 0.4 * p for u, p in zip(uniform, probs)]

    return random.choices(range(NUM_EDGE_TYPES), weights=probs, k=1)[0]


def generate_graph(num_nodes: int, num_edges: int, is_attack: bool) -> GraphData:
    """
    Generate a single synthetic provenance graph with structure-aware edges.

    Key difference from naive random generation: edge types depend on the
    (src_type, dst_type) pair, mimicking real provenance graph structure.
    The model learns these conditional distributions during training.
    Attack graphs break these distributions, producing higher anomaly scores.
    """
    # Assign node types
    node_types = torch.tensor(
        random.choices(range(NUM_NODE_TYPES), weights=BENIGN_TYPE_WEIGHTS, k=num_nodes),
        dtype=torch.long
    )

    # Build index of nodes per type for efficient edge generation
    type_indices = {}
    for ntype in range(NUM_NODE_TYPES):
        indices = (node_types == ntype).nonzero(as_tuple=True)[0].tolist()
        if indices:
            type_indices[ntype] = indices

    # Generate type-specific features, zero-padded to max_feat_dim
    x = torch.zeros(num_nodes, MAX_FEAT_DIM)
    for ntype in range(NUM_NODE_TYPES):
        mask = (node_types == ntype)
        n = mask.sum().item()
        if n > 0:
            feat_dim = NODE_FEATURE_DIMS[ntype]
            x[mask, :feat_dim] = torch.randn(n, feat_dim)

    # Generate structure-aware edges
    pair_list = list(BENIGN_PAIR_WEIGHTS.keys())
    pair_weights_list = list(BENIGN_PAIR_WEIGHTS.values())

    src_list, dst_list, etype_list = [], [], []
    for _ in range(num_edges):
        # Sample a (src_type, dst_type) pair
        pair = random.choices(pair_list, weights=pair_weights_list, k=1)[0]
        st, dt = pair

        # Pick random nodes of the correct types (with fallback)
        if st in type_indices and dt in type_indices:
            s = random.choice(type_indices[st])
            d = random.choice(type_indices[dt])
        else:
            s = random.randint(0, num_nodes - 1)
            d = random.randint(0, num_nodes - 1)

        src_list.append(s)
        dst_list.append(d)
        etype_list.append(_sample_edge_type(st, dt, is_attack))

    edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
    edge_type = torch.tensor(etype_list, dtype=torch.long)

    # Attack perturbation: shift features on ~20% of nodes
    if is_attack:
        anomaly_mask = torch.rand(num_nodes) < 0.2
        x[anomaly_mask] += torch.randn_like(x[anomaly_mask]) * 1.5

    return GraphData(
        x=x, edge_index=edge_index, edge_type=edge_type,
        node_types=node_types, y=int(is_attack),
        num_nodes=num_nodes, num_edges=num_edges
    )


def generate_dataset(n_benign: int = 500, n_attack: int = 100) -> List[GraphData]:
    """Generate a StreamSpot-like dataset of 600 provenance graphs."""
    graphs = []
    for _ in range(n_benign):
        nn = random.randint(50, 500)
        ne = random.randint(nn * 2, nn * 4)  # Provenance graphs are edge-dense
        graphs.append(generate_graph(nn, ne, is_attack=False))

    for _ in range(n_attack):
        nn = random.randint(50, 500)
        ne = random.randint(nn * 2, nn * 4)
        graphs.append(generate_graph(nn, ne, is_attack=True))

    random.shuffle(graphs)
    return graphs


# ── Generate ──────────────────────────────────────────────────────────────────
dataset = generate_dataset()
benign_graphs = [g for g in dataset if g.y == 0]
attack_graphs = [g for g in dataset if g.y == 1]

print(f"Dataset: {len(dataset)} graphs ({len(benign_graphs)} benign, {len(attack_graphs)} attack)")
print(f"Node range: {min(g.num_nodes for g in dataset)}-{max(g.num_nodes for g in dataset)}")
print(f"Edge range: {min(g.num_edges for g in dataset)}-{max(g.num_edges for g in dataset)}")
print(f"Node types: {NUM_NODE_TYPES}, Edge types: {NUM_EDGE_TYPES}")
print(f"Feature dims: {NODE_FEATURE_DIMS}")

Dataset: 600 graphs (500 benign, 100 attack)
Node range: 50-499
Edge range: 111-1882
Node types: 6, Edge types: 9
Feature dims: {0: 8, 1: 4, 2: 6, 3: 5, 4: 4, 5: 3}


---
## Section 2 — Architecture Instantiation

### Component overview

The encoder-decoder architecture follows a four-stage pipeline, each stage motivated by
a specific requirement of provenance-based threat detection:

| Stage | Component | Input → Output | Why |
|-------|-----------|---------------|-----|
| 1 | `NodeTypeProjection` | [N, 3-8] → [N, 64] | Heterogeneous node types have different feature dims; project to common space |
| 2 | `MultiHeadGAT` (1 layer) | [N, 64] → [N, 64] | Spatial context: aggregate neighbour information via learned attention |
| 3 | `GRU Cell` | [N, 64] → [N, 64] | Temporal context: maintain per-node hidden state across events |
| 4 | `EdgeDecoder` (dual-head) | [E, 128] → [E, 1] + [E, 9] | Predict edge existence + edge type for anomaly scoring |

### Key architectural decisions

- **1-layer GAT, not 2:** 97% of E3-THEIA nodes have ≤20 neighbours (ORCHID §5.2).
  A second layer expands receptive field to k²=100 nodes — ~10× computation for
  nodes that have no meaningful 2-hop neighbours.

- **64-dim, not 128:** Halves memory per node (256 → 512 bytes). At E3-THEIA scale
  (690K nodes): 177 MB vs 354 MB. ORCHID finds diminishing returns beyond 64-dim.

- **GRU, not LSTM:** ~25% fewer parameters (no separate cell state). KAIROS validates
  GRU sufficiency for provenance temporal modelling.

- **Dual-head decoder:** Existence head catches *unexpected connections* (e.g., a process
  connecting to a never-seen IP). Type head catches *unexpected operations* (e.g., a
  read-only config file being written to). Both matter for APT detection.

In [3]:
# ── Fallback GAT implementation (pure PyTorch) ────────────────────────────────
# Used when PyG is not available (Windows torch-sparse build failure).
# Mathematically identical to GATConv: α_ij = softmax(LeakyReLU(a^T [Wh_i||Wh_j]))

class GATConvFallback(nn.Module):
    """Single-head GAT layer without torch-sparse dependency."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.W = nn.Linear(in_channels, out_channels, bias=False)
        self.attn = nn.Parameter(torch.empty(2 * out_channels))
        nn.init.xavier_uniform_(self.attn.unsqueeze(0))
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        Wh = self.W(x)
        src, dst = edge_index[0], edge_index[1]

        # Attention coefficients
        edge_feat = torch.cat([Wh[src], Wh[dst]], dim=-1)
        e = self.leaky_relu(edge_feat @ self.attn)

        # Numerically stable softmax per destination node
        e_max = torch.zeros(x.size(0), device=x.device)
        e_max.scatter_reduce_(0, dst, e, reduce='amax', include_self=True)
        alpha = torch.exp(e - e_max[dst])
        alpha_sum = torch.zeros(x.size(0), device=x.device)
        alpha_sum.scatter_add_(0, dst, alpha)
        alpha = alpha / (alpha_sum[dst] + 1e-8)

        # Weighted aggregation
        out = torch.zeros_like(Wh)
        out.scatter_add_(0, dst.unsqueeze(-1).expand_as(Wh[src]), alpha.unsqueeze(-1) * Wh[src])
        return out


class MultiHeadGAT(nn.Module):
    """
    Multi-head GAT: 4 heads × 16-dim = 64-dim output.
    Uses PyG GATConv if available, otherwise falls back to pure PyTorch.
    """

    def __init__(self, in_channels: int, out_per_head: int, num_heads: int):
        super().__init__()
        self.num_heads = num_heads
        if PYGEOMETRIC_AVAILABLE:
            self.gat = GATConv(in_channels, out_per_head, heads=num_heads, concat=True)
        else:
            self.heads = nn.ModuleList([
                GATConvFallback(in_channels, out_per_head) for _ in range(num_heads)
            ])
        self.norm = nn.LayerNorm(out_per_head * num_heads)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        if PYGEOMETRIC_AVAILABLE:
            out = self.gat(x, edge_index)
        else:
            out = torch.cat([h(x, edge_index) for h in self.heads], dim=-1)
        return F.elu(self.norm(out))


# ── Model Components ──────────────────────────────────────────────────────────

class NodeTypeProjection(nn.Module):
    """Per-type linear projection: heterogeneous features → 64-dim common space."""

    def __init__(self, feature_dims: dict, embed_dim: int = 64):
        super().__init__()
        self.embed_dim = embed_dim
        self.projections = nn.ModuleDict({
            str(k): nn.Linear(v, embed_dim) for k, v in feature_dims.items()
        })

    def forward(self, x: torch.Tensor, node_types: torch.Tensor) -> torch.Tensor:
        out = torch.zeros(x.size(0), self.embed_dim, device=x.device)
        for type_id, proj in self.projections.items():
            mask = (node_types == int(type_id))
            if mask.any():
                feat_dim = proj.in_features
                out[mask] = proj(x[mask, :feat_dim])
        return out


class TemporalEncoder(nn.Module):
    """GAT (spatial) + GRU (temporal) + linear projection."""

    def __init__(self, embed_dim: int = 64, gat_heads: int = 4, gat_head_dim: int = 16):
        super().__init__()
        self.gat = MultiHeadGAT(embed_dim, gat_head_dim, gat_heads)
        self.gru = nn.GRUCell(embed_dim, embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x, edge_index, h=None):
        spatial = self.gat(x, edge_index)
        if h is None:
            h = torch.zeros_like(spatial)
        h_new = self.gru(spatial, h)
        z = self.proj(h_new)
        return z, h_new


class EdgeDecoder(nn.Module):
    """
    Dual-head MLP decoder.
    Head 1: P(edge_exists) — sigmoid
    Head 2: P(edge_type)   — 9-class log-softmax
    """

    def __init__(self, embed_dim: int = 64, num_edge_types: int = 9, dropout: float = 0.1):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim * 2),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ELU(),
        )
        self.head_exists = nn.Linear(embed_dim, 1)
        self.head_type = nn.Linear(embed_dim, num_edge_types)

    def forward(self, z, edge_index):
        src, dst = edge_index[0], edge_index[1]
        edge_feat = torch.cat([z[src], z[dst]], dim=-1)
        trunk_out = self.trunk(edge_feat)
        p_exists = self.head_exists(trunk_out)
        p_type = F.log_softmax(self.head_type(trunk_out), dim=-1)
        return p_exists, p_type


class CausalityEncoder(nn.Module):
    """Complete encoder-decoder: features → embeddings → edge predictions."""

    def __init__(self, feature_dims, embed_dim=64, gat_heads=4, gat_head_dim=16,
                 num_edge_types=9, dropout=0.1):
        super().__init__()
        self.node_proj = NodeTypeProjection(feature_dims, embed_dim)
        self.encoder = TemporalEncoder(embed_dim, gat_heads, gat_head_dim)
        self.decoder = EdgeDecoder(embed_dim, num_edge_types, dropout)

    def forward(self, x, node_types, edge_index, h=None):
        emb = self.node_proj(x, node_types)
        z, h_new = self.encoder(emb, edge_index, h)
        p_exists, p_type = self.decoder(z, edge_index)
        return p_exists, p_type, h_new


# ── Instantiate & verify ──────────────────────────────────────────────────────
model = CausalityEncoder(
    feature_dims=NODE_FEATURE_DIMS,
    embed_dim=EMBEDDING_DIM,
    gat_heads=4,
    gat_head_dim=16,
    num_edge_types=NUM_EDGE_TYPES,
    dropout=0.1,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Target: ~60K (64-dim architecture)")
print()

# Verify shapes with a single forward pass
test_graph = dataset[0]
with torch.no_grad():
    p_exists, p_type, h_new = model(
        test_graph.x.to(DEVICE),
        test_graph.node_types.to(DEVICE),
        test_graph.edge_index.to(DEVICE),
    )

print(f"Input:     x={list(test_graph.x.shape)}, edge_index={list(test_graph.edge_index.shape)}")
print(f"p_exists:  {list(p_exists.shape)}  (expected [{test_graph.num_edges}, 1])")
print(f"p_type:    {list(p_type.shape)}  (expected [{test_graph.num_edges}, {NUM_EDGE_TYPES}])")
print(f"h_new:     {list(h_new.shape)}  (expected [{test_graph.num_nodes}, {EMBEDDING_DIM}])")

assert p_exists.shape == (test_graph.num_edges, 1), f"p_exists shape mismatch"
assert p_type.shape == (test_graph.num_edges, NUM_EDGE_TYPES), f"p_type shape mismatch"
assert h_new.shape == (test_graph.num_nodes, EMBEDDING_DIM), f"h_new shape mismatch"
print("\nShape verification PASSED")

Total parameters:     61,194
Trainable parameters: 61,194
Target: ~60K (64-dim architecture)

Input:     x=[443, 8], edge_index=[2, 1087]
p_exists:  [1087, 1]  (expected [1087, 1])
p_type:    [1087, 9]  (expected [1087, 9])
h_new:     [443, 64]  (expected [443, 64])

Shape verification PASSED


---
## Section 3 — Training Loop (Phase 1 Pretrain)

### Why self-supervised?

The model is trained **only on benign graphs** with no attack labels. This is the standard
approach for provenance-based anomaly detection (KAIROS, MAGIC, ORCHID all use it) because:

1. **Benign data is abundant.** Every endpoint generates benign system traces continuously.
   Attack data is rare, expensive to label, and often classified.

2. **Novel attack detection.** A supervised classifier can only detect attack patterns it was
   trained on. An anomaly detector flags anything that deviates from learned benign behaviour,
   including zero-day attacks it has never seen.

3. **The loss function is meaningful.** BCE on edge existence measures "did the model expect
   this connection?" CE on edge type measures "did the model expect *this kind* of operation?"
   When both losses are low, the model has learned what normal system behaviour looks like.
   When they spike on a new edge, that edge is anomalous.

### What "loss decreases" proves

If training loss decreases over epochs, it proves:
- The architecture can represent provenance graph patterns
- Gradients flow correctly through all four stages (projection → GAT → GRU → decoder)
- The model is learning, not just memorizing (we train on 500 diverse graphs)

If loss does **not** decrease, something is broken — stop and debug before proceeding.

In [4]:
# ── Training configuration ───────────────────────────────────────────────────
EPOCHS = 10
LR = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# Train only on benign graphs — self-supervised anomaly detection.
train_graphs = benign_graphs

# ── Loss functions ────────────────────────────────────────────────────────────
# BCE: did the model expect this edge to exist?
# CE:  did the model expect this edge type?
bce_loss_fn = nn.BCEWithLogitsLoss()
ce_loss_fn = nn.NLLLoss()  # p_type is already log-softmax'd


def compute_loss(model, graph):
    """Compute combined BCE + CE loss for a single graph."""
    x = graph.x.to(DEVICE)
    node_types = graph.node_types.to(DEVICE)
    edge_index = graph.edge_index.to(DEVICE)
    edge_type = graph.edge_type.to(DEVICE)

    p_exists, p_type, _ = model(x, node_types, edge_index)

    # All edges in the graph exist (positive samples).
    # In a full system, we'd also sample negative edges. For validation,
    # training on positives alone is sufficient to prove the architecture works.
    exists_target = torch.ones(graph.num_edges, 1, device=DEVICE)

    loss_bce = bce_loss_fn(p_exists, exists_target)
    loss_ce = ce_loss_fn(p_type, edge_type)

    return loss_bce + loss_ce, loss_bce.item(), loss_ce.item()


# ── Training loop ────────────────────────────────────────────────────────────
print(f"Training on {len(train_graphs)} benign graphs for {EPOCHS} epochs")
print(f"Optimizer: Adam, lr={LR}")
print(f"Loss: BCE(edge_exists) + CE(edge_type)")
print("─" * 60)

epoch_losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    total_bce = 0.0
    total_ce = 0.0

    # Shuffle training order each epoch
    indices = list(range(len(train_graphs)))
    random.shuffle(indices)

    for idx in indices:
        graph = train_graphs[idx]
        optimizer.zero_grad()
        loss, bce_val, ce_val = compute_loss(model, graph)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_bce += bce_val
        total_ce += ce_val

    avg_loss = total_loss / len(train_graphs)
    avg_bce = total_bce / len(train_graphs)
    avg_ce = total_ce / len(train_graphs)
    epoch_losses.append(avg_loss)

    print(f"Epoch {epoch+1:2d}/{EPOCHS}  |  Loss: {avg_loss:.4f}  "
          f"(BCE: {avg_bce:.4f}, CE: {avg_ce:.4f})")

# ── Critical assertion: loss must decrease ────────────────────────────────────
print("─" * 60)
print(f"First epoch loss: {epoch_losses[0]:.4f}")
print(f"Last epoch loss:  {epoch_losses[-1]:.4f}")
print(f"Reduction:        {epoch_losses[0] - epoch_losses[-1]:.4f}")

assert epoch_losses[-1] < epoch_losses[0], (
    f"TRAINING FAILED: Loss did not decrease! "
    f"First={epoch_losses[0]:.4f}, Last={epoch_losses[-1]:.4f}. "
    f"The architecture has a bug — do not proceed to Phase 2."
)
print("\nASSERTION PASSED: Training loss decreased — architecture is learning.")

Training on 500 benign graphs for 10 epochs
Optimizer: Adam, lr=0.001
Loss: BCE(edge_exists) + CE(edge_type)
────────────────────────────────────────────────────────────


Epoch  1/10  |  Loss: 1.7358  (BCE: 0.0130, CE: 1.7228)


Epoch  2/10  |  Loss: 1.4383  (BCE: 0.0002, CE: 1.4381)


Epoch  3/10  |  Loss: 1.4122  (BCE: 0.0002, CE: 1.4120)


Epoch  4/10  |  Loss: 1.3963  (BCE: 0.0002, CE: 1.3962)


...

Epoch  6/10  |  Loss: 1.3822  (BCE: 0.0001, CE: 1.3821)


Epoch  7/10  |  Loss: 1.3765  (BCE: 0.0001, CE: 1.3765)


Epoch  8/10  |  Loss: 1.3732  (BCE: 0.0000, CE: 1.3732)


Epoch  9/10  |  Loss: 1.3684  (BCE: 0.0000, CE: 1.3684)


Epoch 10/10  |  Loss: 1.3623  (BCE: 0.0000, CE: 1.3623)
────────────────────────────────────────────────────────────
First epoch loss: 1.7358
Last epoch loss:  1.3623
Reduction:        0.3735

ASSERTION PASSED: Training loss decreased — architecture is learning.


---
## Section 4 — Per-Edge Anomaly Score Computation

### Anomaly scoring pipeline

After training on benign data, the model has learned what "normal" edges look like.
Anomaly scoring works by measuring how much each edge deviates from learned normality:

```
raw_score = BCE(-log P(exists)) + CE(-log P(correct_type))
final_score = raw_score × IDF(src_type, edge_type, dst_type)
```

### What IDF does

IDF (Inverse Document Frequency) weights rare edge patterns higher than common ones.
This is borrowed from CAPTAIN (arXiv:2404.14720): a `process→WRITE→file` edge is so
common that even a high reconstruction error might be noise. But a `socket→EXECUTE→process`
edge is rare enough that any reconstruction error is meaningful.

IDF is computed from training data as:
```
IDF(src_type, edge_type, dst_type) = log(N / (1 + count(src_type, edge_type, dst_type)))
```
where N is total number of edges in training data.

### Why top-10% aggregation

APT attacks affect a small fraction of edges in a graph. A mean over all edges would
dilute the attack signal. Taking the mean of the top-10% highest-scoring edges per graph
focuses on the most anomalous subgraph — the attack footprint.

### What the plot should show

Attack graph scores should be *statistically higher* than benign graph scores. Perfect
separation is not expected on synthetic data — we just need better-than-random to confirm
the scoring pipeline works. Perfect separation comes after training on real DARPA TC data.

In [5]:
# ── Compute IDF weights from training data ───────────────────────────────────

def compute_idf_weights(graphs: List[GraphData]) -> Dict[str, float]:
    """
    Compute IDF weights for (src_type, edge_type, dst_type) triples.
    Rare patterns get higher weight — borrowed from CAPTAIN's approach.
    """
    triple_counts = Counter()
    total_edges = 0

    for g in graphs:
        src_nodes = g.edge_index[0]
        dst_nodes = g.edge_index[1]
        for i in range(g.num_edges):
            src_type = g.node_types[src_nodes[i]].item()
            dst_type = g.node_types[dst_nodes[i]].item()
            etype = g.edge_type[i].item()
            triple_counts[f"{src_type}_{etype}_{dst_type}"] += 1
            total_edges += 1

    idf = {}
    for triple, count in triple_counts.items():
        idf[triple] = math.log(total_edges / (1 + count))

    return idf


print("Computing IDF weights from training data...")
idf_weights = compute_idf_weights(train_graphs)
print(f"Unique (src_type, edge_type, dst_type) triples: {len(idf_weights)}")

# Show top-5 highest IDF (rarest patterns) and top-5 lowest (most common)
sorted_idf = sorted(idf_weights.items(), key=lambda x: x[1], reverse=True)
print(f"\nRarest patterns (highest IDF):")
for triple, w in sorted_idf[:5]:
    print(f"  {triple}: {w:.3f}")
print(f"Most common patterns (lowest IDF):")
for triple, w in sorted_idf[-5:]:
    print(f"  {triple}: {w:.3f}")

Computing IDF weights from training data...


Unique (src_type, edge_type, dst_type) triples: 86

Rarest patterns (highest IDF):
  0_4_4: 12.203
  1_0_3: 12.203
  1_5_3: 12.203
  4_6_2: 12.203
  3_6_1: 12.203
Most common patterns (lowest IDF):
  2_6_0: 2.812
  1_2_0: 2.803
  0_0_1: 2.414
  0_3_0: 2.409
  0_1_1: 2.256


In [6]:
# ── Compute per-edge anomaly scores for all graphs ───────────────────────────

def score_graph(model, graph, idf_weights, device):
    """
    Compute anomaly score for a single graph.

    Returns:
        graph_score: mean of top-10% edge scores (float)
        edge_scores: all per-edge scores (list of floats)
    """
    model.eval()
    with torch.no_grad():
        x = graph.x.to(device)
        node_types = graph.node_types.to(device)
        edge_index = graph.edge_index.to(device)
        edge_type = graph.edge_type.to(device)

        p_exists, p_type, _ = model(x, node_types, edge_index)

        # Per-edge raw anomaly score: BCE + CE
        exists_target = torch.ones(graph.num_edges, 1, device=device)
        bce_per_edge = F.binary_cross_entropy_with_logits(
            p_exists, exists_target, reduction='none'
        ).squeeze(-1)  # [E]

        ce_per_edge = F.nll_loss(
            p_type, edge_type, reduction='none'
        )  # [E]

        raw_scores = bce_per_edge + ce_per_edge  # [E]

        # Apply IDF weighting
        src_nodes = graph.edge_index[0]
        dst_nodes = graph.edge_index[1]
        idf_multipliers = torch.ones(graph.num_edges, device=device)
        for i in range(graph.num_edges):
            src_t = graph.node_types[src_nodes[i]].item()
            dst_t = graph.node_types[dst_nodes[i]].item()
            et = graph.edge_type[i].item()
            key = f"{src_t}_{et}_{dst_t}"
            idf_multipliers[i] = idf_weights.get(key, math.log(graph.num_edges))

        final_scores = raw_scores * idf_multipliers  # [E]

        # Aggregate: mean of top-10% edge scores
        k = max(1, int(0.10 * graph.num_edges))
        top_scores, _ = torch.topk(final_scores, k)
        graph_score = top_scores.mean().item()

        return graph_score, final_scores.cpu().tolist()


# Score all graphs
print("Scoring all graphs...")
benign_scores = []
attack_scores = []

for g in dataset:
    score, _ = score_graph(model, g, idf_weights, DEVICE)
    if g.y == 0:
        benign_scores.append(score)
    else:
        attack_scores.append(score)

print(f"\nBenign scores: mean={np.mean(benign_scores):.4f}, "
      f"std={np.std(benign_scores):.4f}, "
      f"median={np.median(benign_scores):.4f}")
print(f"Attack scores: mean={np.mean(attack_scores):.4f}, "
      f"std={np.std(attack_scores):.4f}, "
      f"median={np.median(attack_scores):.4f}")
print(f"\nSeparation (attack_mean - benign_mean): {np.mean(attack_scores) - np.mean(benign_scores):.4f}")

Scoring all graphs...



Benign scores: mean=12.9804, std=2.5541, median=12.7484
Attack scores: mean=124.5101, std=6.8488, median=125.5340

Separation (attack_mean - benign_mean): 111.5297


In [7]:
# ── Visualize score distributions ────────────────────────────────────────────
# Using text-based histogram since matplotlib may not be available in all envs.
# If matplotlib is available, we use it for a proper plot.

try:
    import matplotlib
    matplotlib.use('Agg')  # Non-interactive backend for notebook compatibility
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

if HAS_MPL:
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    ax.hist(benign_scores, bins=30, alpha=0.6, label=f'Benign (n={len(benign_scores)})', color='steelblue')
    ax.hist(attack_scores, bins=30, alpha=0.6, label=f'Attack (n={len(attack_scores)})', color='firebrick')
    ax.set_xlabel('Graph-Level Anomaly Score (mean of top-10% edge scores)')
    ax.set_ylabel('Count')
    ax.set_title('Phase 1: Anomaly Score Distribution — Benign vs Attack')
    ax.legend()
    ax.axvline(np.mean(benign_scores), color='steelblue', linestyle='--', alpha=0.8)
    ax.axvline(np.mean(attack_scores), color='firebrick', linestyle='--', alpha=0.8)
    plt.tight_layout()
    plt.show()
    print("Plot rendered above.")
else:
    # Text-based fallback
    print("\nScore Distribution (text histogram):")
    all_scores = benign_scores + attack_scores
    min_s, max_s = min(all_scores), max(all_scores)
    n_bins = 20
    bin_width = (max_s - min_s) / n_bins if max_s > min_s else 1.0

    for label, scores, marker in [("Benign", benign_scores, "#"), ("Attack", attack_scores, "*")]:
        print(f"\n  {label}:")
        bins = [0] * n_bins
        for s in scores:
            b = min(int((s - min_s) / bin_width), n_bins - 1)
            bins[b] += 1
        max_count = max(bins) if max(bins) > 0 else 1
        for i, count in enumerate(bins):
            bar = marker * int(40 * count / max_count)
            lo = min_s + i * bin_width
            print(f"  {lo:7.2f} | {bar}  ({count})")

# ── Separation check ─────────────────────────────────────────────────────────
mean_diff = np.mean(attack_scores) - np.mean(benign_scores)
print(f"\nMean score difference (attack - benign): {mean_diff:.4f}")

if mean_diff > 0:
    print("CHECK PASSED: Attack graphs score higher than benign graphs on average.")
    print("The anomaly scoring pipeline is working as expected.")
else:
    print("WARNING: Attack graphs do NOT score higher than benign graphs.")
    print("This may indicate the model needs more training or the synthetic data")
    print("does not have sufficient structural difference between classes.")
    print("On synthetic data, this is acceptable — real DARPA TC data will provide")
    print("stronger signal. Proceeding with checkpoint save.")

Plot rendered above.

Mean score difference (attack - benign): 111.5297
CHECK PASSED: Attack graphs score higher than benign graphs on average.
The anomaly scoring pipeline is working as expected.


C:\Users\juda\AppData\Local\Temp\ipykernel_37884\947128408.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 5 — Checkpoint Save

Save two artifacts for the next phase:
1. **Model state_dict** — all learned weights, loadable with `model.load_state_dict()`
2. **IDF weights** — precomputed from training data, used for anomaly scoring at inference

Phase 2 will load these and fine-tune on DARPA TC E3 data with incremental alignment.

In [8]:
# ── Save checkpoints ─────────────────────────────────────────────────────────
import pathlib

# Determine checkpoint directory relative to this notebook
# Works both when run from notebook dir and from repo root
notebook_dir = pathlib.Path(".").resolve()
# Navigate to causality-engine/checkpoints regardless of cwd
repo_root = notebook_dir
while repo_root.name and not (repo_root / '.git').exists():
    repo_root = repo_root.parent
checkpoint_dir = repo_root / 'causality-engine' / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# 1. Model state_dict
model_path = checkpoint_dir / 'phase1_streamspot.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'epoch': EPOCHS,
    'final_loss': epoch_losses[-1],
    'embed_dim': EMBEDDING_DIM,
    'num_edge_types': NUM_EDGE_TYPES,
    'feature_dims': NODE_FEATURE_DIMS,
    'gat_heads': 4,
    'gat_head_dim': 16,
}, model_path)
print(f"Model saved to: {model_path}")
print(f"  Size: {model_path.stat().st_size / 1024:.1f} KB")

# 2. IDF weights
idf_path = checkpoint_dir / 'idf_weights.json'
with open(idf_path, 'w') as f:
    json.dump(idf_weights, f, indent=2)
print(f"IDF weights saved to: {idf_path}")
print(f"  Entries: {len(idf_weights)}")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PHASE 1 COMPLETE — Architecture Validation")
print("=" * 60)
print(f"")
print(f"Architecture:")
print(f"  NodeTypeProjection: 6 types → {EMBEDDING_DIM}-dim")
print(f"  GAT: 1 layer, 4 heads × 16-dim = {EMBEDDING_DIM}-dim")
print(f"  GRU: {EMBEDDING_DIM}-dim hidden state")
print(f"  EdgeDecoder: dual-head (exists + {NUM_EDGE_TYPES}-class type)")
print(f"  Total parameters: {total_params:,}")
print(f"")
print(f"Training:")
print(f"  Data: {len(train_graphs)} benign synthetic graphs")
print(f"  Epochs: {EPOCHS}")
print(f"  Loss: {epoch_losses[0]:.4f} → {epoch_losses[-1]:.4f} (decreased: YES)")
print(f"")
print(f"Anomaly scoring:")
print(f"  Benign mean: {np.mean(benign_scores):.4f}")
print(f"  Attack mean: {np.mean(attack_scores):.4f}")
print(f"  Separation:  {np.mean(attack_scores) - np.mean(benign_scores):.4f}")
print(f"")
print(f"Saved:")
print(f"  {model_path}")
print(f"  {idf_path}")
print(f"")
print(f"Next steps (Phase 2):")
print(f"  1. Load DARPA TC E3 data and convert to CDM graph format")
print(f"  2. Fine-tune on E3 benign subgraphs")
print(f"  3. Evaluate detection on E3 attack campaigns")
print(f"  4. Implement incremental alignment module")

Model saved to: J:\THESIS-EDR\causality-engine\checkpoints\phase1_streamspot.pt
  Size: 252.2 KB
IDF weights saved to: J:\THESIS-EDR\causality-engine\checkpoints\idf_weights.json
  Entries: 86

PHASE 1 COMPLETE — Architecture Validation

Architecture:
  NodeTypeProjection: 6 types → 64-dim
  GAT: 1 layer, 4 heads × 16-dim = 64-dim
  GRU: 64-dim hidden state
  EdgeDecoder: dual-head (exists + 9-class type)
  Total parameters: 61,194

Training:
  Data: 500 benign synthetic graphs
  Epochs: 10
  Loss: 1.7358 → 1.3623 (decreased: YES)

Anomaly scoring:
  Benign mean: 12.9804
  Attack mean: 124.5101
  Separation:  111.5297

Saved:
  J:\THESIS-EDR\causality-engine\checkpoints\phase1_streamspot.pt
  J:\THESIS-EDR\causality-engine\checkpoints\idf_weights.json

Next steps (Phase 2):
  1. Load DARPA TC E3 data and convert to CDM graph format
  2. Fine-tune on E3 benign subgraphs
  3. Evaluate detection on E3 attack campaigns
  4. Implement incremental alignment module
